# Taller Python + IA aplicada al negocio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marlonleandro/taller-python-banca/blob/main/Caso_Practico_02.ipynb)

## Objetivos del Taller

Integrar herramientas de inteligencia artificial para mejorar el análisis de datos:

- **Python** calcula y organiza los datos
- **IA** interpreta los resultados y redacta un reporte ejecutivo

La lógica es:
1. Cargar datos
2. Limpiar
3. Calcular KPIs
4. Enviar resultados a un modelo
5. Obtener narrativa de negocio


## Archivos de datos incluidos

- `dataset_clientes.csv`
- `dataset_creditos.csv`
- `dataset_transacciones.csv`

## Instalar librerias necesarias

```shell
python -m pip install python-dotenv
python -m pip install openai
```

## 1. Importar Librerías


In [1]:
# Importar librerias necesarias
import os
from datetime import date
import pandas as pd


## 2. Carga de datos


In [2]:
clientes = pd.read_csv("./data/dataset_clientes.csv")
creditos = pd.read_csv("./data/dataset_creditos.csv", parse_dates=["fecha_vencimiento"])
transacc = pd.read_csv("./data/dataset_transacciones.csv", parse_dates=["fecha_hora"])

base = clientes.merge(creditos, on="cliente_id", how="left")

print(f"Registros: {len(base)} | Columnas: {len(base.columns)}")
display(base.head())


Registros: 815 | Columnas: 15


,cliente_id,nombre,segmento,fecha_apertura,estado,edad,ingresos,zona_geografica,credito_id,tipo_credito,saldo_actual,fecha_vencimiento,dias_mora,tasa_interes,ratio_deuda_ingreso
0,C00001,Lucía León,Microempresa,2024-02-16,Inactive,37,11962.79,Centro,NaN,NaN,NaN,NaT,NaN,NaN,NaN
1,C00002,Carla Rojas,Pyme,2020-10-16,Active,23,23024.90,Lima,CR000304,Capital de Trabajo,103574.75,2026-05-22,0.0,0.2440,0.3749
2,C00002,Carla Rojas,Pyme,2020-10-16,Active,23,23024.90,Lima,CR000645,Capital de Trabajo,61388.26,2026-05-16,0.0,0.1270,0.2222
3,C00003,Jorge Salas,Pyme,2020-10-10,Active,55,16248.68,Oriente,NaN,NaN,NaN,NaT,NaN,NaN,NaN
4,C00004,Carla Rojas,Microempresa,2025-12-08,Active,63,10364.31,Centro,CR000092,Capital de Trabajo,18776.24,2026-04-29,0.0,0.3776,0.1510


## 3. Limpieza


In [3]:
creditos["saldo_actual"] = pd.to_numeric(creditos["saldo_actual"], errors="coerce")
creditos["fecha_vencimiento"] = pd.to_datetime(creditos["fecha_vencimiento"], errors="coerce")
base["saldo_actual"] = pd.to_numeric(base["saldo_actual"], errors="coerce")

base = base.dropna(subset=["cliente_id", "saldo_actual"])
base["segmento"] = base["segmento"].astype(str).str.upper().str.strip()

print(f"Registros limpios: {len(base)}")
display(base[["cliente_id", "segmento", "saldo_actual"]].head(3))


Registros limpios: 720


,cliente_id,segmento,saldo_actual
1,C00002,PYME,103574.75
2,C00002,PYME,61388.26
4,C00004,MICROEMPRESA,18776.24


## 4. KPIs con Python


In [4]:
ref = pd.Timestamp(date.today())

creditos["dpd"] = (ref - creditos["fecha_vencimiento"]).dt.days.clip(lower=0)
creditos["es_moroso"] = creditos["dpd"] > 0

base = base.merge(
    creditos[["credito_id", "dpd", "es_moroso"]],
    on="credito_id",
    how="left"
)

kpis = base.groupby("segmento").agg(
    portafolio=("saldo_actual", "sum"),
    clientes=("cliente_id", "nunique"),
    mora_pct=("es_moroso", "mean"),
    dpd_avg=("dpd", "mean"),
    ticket_prom=("saldo_actual", "mean"),
).reset_index()

kpis["mora_pct"] = (kpis["mora_pct"] * 100).round(1)
kpis["alerta"] = kpis["mora_pct"] > 8.0

top_riesgo = creditos.nlargest(10, "dpd")[["cliente_id", "saldo_actual", "dpd"]]
vol_canal = transacc.groupby("canal")["monto"].agg(total="sum", n="count").reset_index()

display(kpis)
display(top_riesgo)
display(vol_canal)


,segmento,portafolio,clientes,mora_pct,dpd_avg,ticket_prom,alerta
0,MICROEMPRESA,3560984.07,69,16.8,6.650350,24901.986503,True
1,PREMIUM,13697845.95,44,4.5,1.617978,153908.381461,False
2,PYME,11225940.06,79,9.0,3.305085,63423.390169,True
3,RETAIL,5435800.05,163,7.4,3.299035,17478.456752,False


,cliente_id,saldo_actual,dpd
569,C00023,39313.08,118
69,C00011,28604.63,112
245,C00308,78987.76,101
199,C00213,9360.67,100
9,C00322,41252.12,96
82,C00098,23877.17,95
271,C00102,3341.71,95
114,C00395,27834.37,92
204,C00035,42193.93,91
596,C00284,10317.60,89


,canal,total,n
0,ATM,262790.25,368
1,App,515984.50,818
2,POS,608117.85,954
3,Web,306311.58,460


### Guardamos datos (opcional)

In [8]:
# Guardamos outputs intermedios
kpis.to_csv("./results/kpis_segmento.csv", index=False)
top_riesgo.to_csv("./results/top_riesgo.csv", index=False)
vol_canal.to_csv("./results/volumen_canal.csv", index=False)

## 5. Preparar contexto para la IA

Aquí convertimos tablas a texto para que el modelo pueda interpretarlas.


In [5]:
tabla_kpis = kpis.to_string(index=False)
alertas = kpis[kpis["alerta"]][["segmento", "mora_pct", "portafolio"]].to_string(index=False)
tabla_top_riesgo = top_riesgo.to_string(index=False)
tabla_vol_canal = vol_canal.to_string(index=False)

prompt = f'''
Eres analista de riesgo bancario senior.

KPIs de cartera actuales:
{tabla_kpis}

Segmentos en alerta (mora > 8%):
{alertas}

Top 10 clientes por días de mora:
{tabla_top_riesgo}

Volumen transaccional por canal:
{tabla_vol_canal}

Genera un reporte ejecutivo con esta estructura:
1. Resumen de situación (máximo 3 oraciones)
2. Alertas críticas + acción recomendada
3. Oportunidades de negocio
4. Próximos pasos para el equipo

Usa lenguaje ejecutivo, claro y conciso.
'''

print(prompt)



Eres analista de riesgo bancario senior.

KPIs de cartera actuales:
    segmento  portafolio  clientes  mora_pct  dpd_avg   ticket_prom  alerta
MICROEMPRESA  3560984.07        69      16.8 6.650350  24901.986503    True
     PREMIUM 13697845.95        44       4.5 1.617978 153908.381461   False
        PYME 11225940.06        79       9.0 3.305085  63423.390169    True
      RETAIL  5435800.05       163       7.4 3.299035  17478.456752   False

Segmentos en alerta (mora > 8%):
    segmento  mora_pct  portafolio
MICROEMPRESA      16.8  3560984.07
        PYME       9.0 11225940.06

Top 10 clientes por días de mora:
cliente_id  saldo_actual  dpd
    C00023      39313.08  118
    C00011      28604.63  112
    C00308      78987.76  101
    C00213       9360.67  100
    C00322      41252.12   96
    C00098      23877.17   95
    C00102       3341.71   95
    C00395      27834.37   92
    C00035      42193.93   91
    C00284      10317.60   89

Volumen transaccional por canal:
canal     tot

## 6. Configurar la API key

Cre un archivo .env con el siguiente contenido:

```shell
# .env
OPENAI_API_KEY=sk-proj-xxxxxxxxxxxxxxxx
```

Puedes obtener tu API KEY de OpenAI en la siguiente dirección:
https://platform.openai.com/

## 7. Llamar al modelo con OpenAI

Este es el paso donde la IA entra realmente al flujo.


In [6]:
import dotenv
from openai import OpenAI
dotenv.load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

reporte_ia = response.output_text
print(reporte_ia)


### Reporte Ejecutivo

#### 1. Resumen de Situación
La cartera presenta segmentos en riesgo, destacando a las Microempresas y PYMEs con tasas de mora superiores al 8%. A pesar de la salud de otros segmentos como Premium y Retail, la mora incrementada en los clientes críticos podría impactar negativamente en la calidad del portafolio. El indicador de días de mora revela un grupo de clientes con saldos significativos, resaltando la necesidad de atención inmediata.

#### 2. Alertas Críticas + Acción Recomendada
- **Microempresas:** Mora del 16.8% (portafolio: 3,560,984.07). 
- **PYMEs:** Mora del 9.0% (portafolio: 11,225,940.06).
  
**Acción Recomendada:** Implementar un plan de recuperación de crédito centrado en negociación y reestructuración de deudas. Se sugiere priorizar contacto con los 10 principales clientes con mayores días de mora para mitigar el riesgo inmediato.

#### 3. Oportunidades de Negocio
Con el crecimiento del uso de canales digitales, especialmente en la App y POS, ex

## 8. Exportar el reporte ejecutivo generado por IA


In [7]:
with open("./results/reporte_ejecutivo_ia.txt", "w", encoding="utf-8") as f:
    f.write(reporte_ia)

print("Archivo guardado: reporte_ejecutivo_ia.txt")


Archivo guardado: reporte_ejecutivo_ia.txt
